# ROGII - Wellbore Geology Prediction — submission notebook

Self-contained: no internet access needed (Code Competition requirement). Stage 2
baseline — per-well linear prior `tvt ~ MD + Z`, fit on each well's known rows and
predicted on that well's real evaluation zone (`TVT_input` is `NaN` there).

Locally verified (773 train wells): overall RMSE 67.09, median per-well 33.07 — far off
the ~4.86 public-LB leader. This is a **pipeline-proving baseline**, not a competitive
model — ships first so the submit path works end to end; see
`../context/05-plan-of-attack.md` for Stage 3+ (typewell/GR alignment, CNN) which is what
actually closes the gap.

`id = {WELLNAME}_{row_index}` where `row_index` is the 0-based row position in that
well's `__horizontal_well.csv` (confirmed against `sample_submission.csv`).

In [ ]:
import glob
import os

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# Kaggle mounts competition data at /kaggle/input/competitions/<slug>/ for
# notebook submissions (confirmed from a live Kaggle session's own os.walk output -
# NOT /kaggle/input/<slug>/, which is the path used for attached Datasets). Fall
# back to the local dev copy (../data) so this notebook also runs outside Kaggle.
_KAGGLE_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction"
DATA_DIR = _KAGGLE_DIR if os.path.isdir(_KAGGLE_DIR) else os.path.join("..", "data")
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")
SAMPLE_SUBMISSION = os.path.join(DATA_DIR, "sample_submission.csv")

print(f"DATA_DIR = {DATA_DIR}")
assert os.path.isdir(TRAIN_DIR), f"train dir not found: {TRAIN_DIR}"
assert os.path.isdir(TEST_DIR), f"test dir not found: {TEST_DIR}"

In [ ]:
def list_wells(split_dir):
    files = glob.glob(os.path.join(split_dir, "*__horizontal_well.csv"))
    return sorted(os.path.basename(f).split("__")[0] for f in files)


def fit_predict_well(df, global_fallback):
    """Fit tvt ~ MD + Z on known rows, predict eval-zone rows.

    Returns (eval_positional_index, preds) where eval_positional_index is the
    0-based row position in df (matches the submission id's row_index). Never
    raises - a competition notebook that dies partway through writes NOTHING,
    which is worse than a degraded prediction for one well.
    """
    df = df.reset_index(drop=True)
    known = df[df["TVT_input"].notna() & df["MD"].notna() & df["Z"].notna()]
    eval_rows = df[df["TVT_input"].isna()]
    if len(eval_rows) == 0:
        return np.array([], dtype=int), np.array([])

    eval_valid = eval_rows[eval_rows["MD"].notna() & eval_rows["Z"].notna()]
    eval_bad = eval_rows.index.difference(eval_valid.index)

    if len(known) >= 2 and len(eval_valid) > 0:
        try:
            model = LinearRegression()
            model.fit(known[["MD", "Z"]], known["TVT_input"])
            preds_valid = model.predict(eval_valid[["MD", "Z"]])
        except Exception as exc:  # noqa: BLE001 - degrade, never crash the run
            print(f"  fit/predict failed ({exc}); falling back to known/global mean")
            fallback = known["TVT_input"].mean() if len(known) else global_fallback
            preds_valid = np.full(len(eval_valid), fallback)
    else:
        fallback = known["TVT_input"].mean() if len(known) else global_fallback
        preds_valid = np.full(len(eval_valid), fallback)

    idx = list(eval_valid.index)
    preds = list(preds_valid)
    if len(eval_bad) > 0:
        # Rows with missing MD/Z in the eval zone itself - can't feature them,
        # still must predict something so every required id gets a row.
        idx += list(eval_bad)
        preds += [global_fallback] * len(eval_bad)

    return np.array(idx, dtype=int), np.array(preds)

In [ ]:
test_wells = list_wells(TEST_DIR)
print(f"test wells: {len(test_wells)}")

# A single scalar fallback for the worst case (a well we can't process at all) -
# the mean known TVT_input across a sample of train wells. Cheap, always available,
# never NaN. Better than crashing the whole run over one bad well.
train_wells_sample = list_wells(TRAIN_DIR)[:50]
_fallback_vals = []
for w in train_wells_sample:
    try:
        d = pd.read_csv(os.path.join(TRAIN_DIR, f"{w}__horizontal_well.csv"))
        _fallback_vals.append(d["TVT_input"].mean())
    except Exception:
        pass
GLOBAL_FALLBACK = float(np.nanmean(_fallback_vals)) if _fallback_vals else 0.0
print(f"GLOBAL_FALLBACK = {GLOBAL_FALLBACK:.2f}")

rows = []
failed_wells = []
for well in test_wells:
    try:
        df = pd.read_csv(os.path.join(TEST_DIR, f"{well}__horizontal_well.csv"))
        eval_idx, preds = fit_predict_well(df, GLOBAL_FALLBACK)
        for idx, pred in zip(eval_idx, preds):
            rows.append({"id": f"{well}_{idx}", "tvt": pred})
    except Exception as exc:  # noqa: BLE001 - one bad well must not kill the run
        print(f"well {well} failed entirely ({exc}); will backfill its ids at the end")
        failed_wells.append(well)

submission = pd.DataFrame(rows, columns=["id", "tvt"])
print(f"built {len(submission)} predictions across {len(test_wells) - len(failed_wells)} wells "
      f"({len(failed_wells)} wells failed: {failed_wells})")
submission.head()

In [ ]:
# Reconcile against the sample submission WITHOUT asserting/crashing - any gap
# (missing id, duplicate id, NaN prediction) gets filled with GLOBAL_FALLBACK and
# printed as a warning instead of killing the run. A degraded-but-complete
# submission.csv beats a perfect run that never finishes and writes nothing.
sample = pd.read_csv(SAMPLE_SUBMISSION)

submission = submission.drop_duplicates(subset="id", keep="first")

merged = sample[["id"]].merge(submission, on="id", how="left")
n_missing = merged["tvt"].isna().sum()
if n_missing > 0:
    print(f"WARNING: {n_missing} required ids had no prediction - filling with "
          f"GLOBAL_FALLBACK ({GLOBAL_FALLBACK:.2f})")
    merged["tvt"] = merged["tvt"].fillna(GLOBAL_FALLBACK)

extra_ids = set(submission["id"]) - set(sample["id"])
if extra_ids:
    print(f"NOTE: {len(extra_ids)} predicted ids aren't in sample_submission - dropped "
          f"(harmless, e.g. from a visible-example test well not in the real hidden set)")

assert len(merged) == len(sample), f"row count still off: {len(merged)} vs {len(sample)}"
assert merged["tvt"].notna().all(), "still have NaNs after fallback fill - this should be impossible"

submission = merged
print("Reconciliation complete - submission.csv will have every required id.")
submission.to_csv("submission.csv", index=False)
print("Wrote submission.csv:", submission.shape)